# Lab 08 Solution: Iterative Refinement

Build the **Reflect → Iterate** loop that makes AI coding agents powerful —
code gets progressively better through automated feedback cycles.

**What you'll learn:**
- Measuring code quality with automated metrics
- Applying targeted improvements in each iteration
- Building a refinement loop with a stopping condition

No API key needed — pure Python standard library.

In [ ]:
import os
import shutil
import json
import ast
import re

WORKDIR = "/tmp/aidev-lab-02-08"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: The Reflect-Iterate Cycle

In Lab 01, you learned the 5-phase agent loop: **Plan → Code → Test → Reflect → Iterate**.
In Lab 07, you learned AST basics — parsing code, counting functions, and detecting issues.

Now we use those same AST techniques to **measure quality** and **drive an improvement loop**.

The last two phases are where the magic happens:

```
┌─────────────────────────────────────────────┐
│   Generated Code                            │
│        ↓                                    │
│   Quality Check (measure metrics)           │
│        ↓                                    │
│   Score ≥ Target?  ──YES──→  Done!          │
│        │ NO                                 │
│        ↓                                    │
│   Apply Improvement (fix one issue)         │
│        ↓                                    │
│   Loop back to Quality Check                │
│   (max 3 iterations to prevent runaway)     │
└─────────────────────────────────────────────┘
```

Each iteration fixes **one category** of issues. This is how real agents work —
they don't try to fix everything at once.

## Step 2: Quality Metrics

We'll measure code quality across four dimensions:

| Metric | What it checks | Points |
|--------|---------------|--------|
| **Syntax** | Code compiles without errors | 0 or 1 |
| **Docstrings** | % of functions with docstrings | 0.0 – 1.0 |
| **Type hints** | % of functions with return type annotations | 0.0 – 1.0 |
| **Error handling** | Has at least one try/except block | 0 or 1 |

**Quality score** = average of all four metrics (0.0 – 1.0)

In [ ]:
# Starting code: a valid but bare-bones Python module
starting_code = '''def fetch_data(url):
    import urllib.request
    response = urllib.request.urlopen(url)
    return response.read().decode()

def parse_items(raw_data):
    lines = raw_data.strip().split("\\n")
    items = []
    for line in lines:
        parts = line.split(",")
        items.append({"name": parts[0], "value": parts[1]})
    return items

def save_results(items, filepath):
    import json
    with open(filepath, "w") as f:
        json.dump(items, f)
'''

print("Starting code:")
print(starting_code)

## TODO 1 Solution: Implement `code_quality_score`

In [ ]:
def code_quality_score(code):
    """Measure code quality across four dimensions."""
    # Check syntax
    try:
        tree = ast.parse(code)
        syntax = 1
    except SyntaxError:
        return {
            "syntax": 0,
            "docstrings": 0.0,
            "type_hints": 0.0,
            "error_handling": 0,
            "overall": 0.0,
        }

    # Find all functions
    functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
    total_funcs = len(functions)

    if total_funcs == 0:
        docstrings = 1.0
        type_hints = 1.0
    else:
        # Check docstrings
        has_docstring = 0
        for func in functions:
            if (func.body and isinstance(func.body[0], ast.Expr)
                    and isinstance(func.body[0].value, ast.Constant)
                    and isinstance(func.body[0].value.value, str)):
                has_docstring += 1
        docstrings = round(has_docstring / total_funcs, 2)

        # Check type hints (return annotations)
        has_hints = sum(1 for f in functions if f.returns is not None)
        type_hints = round(has_hints / total_funcs, 2)

    # Check error handling
    has_try = any(isinstance(node, ast.Try) for node in ast.walk(tree))
    error_handling = 1 if has_try else 0

    overall = round((syntax + docstrings + type_hints + error_handling) / 4, 2)

    return {
        "syntax": syntax,
        "docstrings": docstrings,
        "type_hints": type_hints,
        "error_handling": error_handling,
        "overall": overall,
    }

In [ ]:
score1 = 0
checks_1 = []

r1 = code_quality_score(starting_code)

if isinstance(r1, dict) and "overall" in r1:
    checks_1.append(("Returns a dict with overall score", "PASS"))
    score1 += 1
else:
    checks_1.append((f"Returns a dict with overall score (got {type(r1).__name__})", "FAIL"))

if isinstance(r1, dict) and r1.get("syntax") == 1:
    checks_1.append(("Detects valid syntax (syntax=1)", "PASS"))
    score1 += 1
else:
    checks_1.append((f"Detects valid syntax (got syntax={r1.get('syntax') if isinstance(r1, dict) else '?'})", "FAIL"))

if isinstance(r1, dict) and r1.get("docstrings") == 0.0:
    checks_1.append(("Measures docstring coverage (0/3 = 0.0)", "PASS"))
    score1 += 1
else:
    checks_1.append((f"Measures docstring coverage (expected 0.0, got {r1.get('docstrings') if isinstance(r1, dict) else '?'})", "FAIL"))

if isinstance(r1, dict) and r1.get("type_hints") == 0.0:
    checks_1.append(("Measures type hint coverage (0/3 = 0.0)", "PASS"))
    score1 += 1
else:
    checks_1.append((f"Measures type hint coverage (expected 0.0, got {r1.get('type_hints') if isinstance(r1, dict) else '?'})", "FAIL"))

for check, status in checks_1:
    print(f"    [{status}] {check}")

print(f"\n  Starting code metrics:")
for k, v in r1.items():
    print(f"    {k}: {v}")

print(f"\n  Score: {score1}/4")

## Step 3: Improvement Strategies

Each iteration applies **one** improvement category:

| Round | Improvement | What changes |
|-------|------------|-------------|
| 1 | Add docstrings | Insert `"""..."""` after each `def` line |
| 2 | Add type hints | Append `-> str` or `-> list` to function signatures |
| 3 | Add error handling | Wrap risky operations in `try/except` |

We use **string transformations** (not AST rewriting) to keep it simple and visual.

## TODO 2 Solution: Implement `improve_code`

In [ ]:
def improve_code(code, improvement_type):
    """Apply one improvement to the code."""

    if improvement_type == "docstrings":
        # Add docstrings to functions that don't have them
        # Match: def name(...):  NOT followed by a docstring
        result = re.sub(
            r'(def (\w+)\([^)]*\)[^:]*:)\n(?!\s+""")',
            r'\1\n    """\2 operation."""\n',
            code,
        )
        return result

    elif improvement_type == "type_hints":
        # Add -> str return annotations to functions without them
        result = re.sub(
            r'def (\w+)\(([^)]*)\):',
            r'def \1(\2) -> str:',
            code,
        )
        return result

    elif improvement_type == "error_handling":
        # Add try/except to the first function that doesn't have one
        if "try:" in code:
            return code  # Already has error handling

        lines = code.split("\n")
        new_lines = []
        i = 0
        wrapped_one = False
        while i < len(lines):
            line = lines[i]
            # Find a function definition
            if line.startswith("def ") and not wrapped_one:
                new_lines.append(line)
                i += 1
                # Collect body lines (indented)
                body_lines = []
                while i < len(lines) and (lines[i].startswith("    ") or lines[i].strip() == ""):
                    if lines[i].strip() == "" and (i + 1 >= len(lines) or not lines[i + 1].startswith("    ")):
                        break
                    body_lines.append(lines[i])
                    i += 1
                # Wrap body in try/except
                new_lines.append("    try:")
                for bl in body_lines:
                    new_lines.append("    " + bl)
                new_lines.append("    except Exception as e:")
                new_lines.append("        return f\"Error: {e}\"")
                wrapped_one = True
            else:
                new_lines.append(line)
                i += 1
        return "\n".join(new_lines)

    return code

In [ ]:
score2 = 0
checks_2 = []

# Test docstrings
r1 = improve_code(starting_code, "docstrings")
try:
    compile(r1, "<string>", "exec")
    has_docstrings = '"""' in r1 and r1.count('"""') >= 2
    if has_docstrings:
        checks_2.append(("Adds docstrings (valid syntax + docstrings present)", "PASS"))
        score2 += 1
    else:
        checks_2.append(("Adds docstrings (no docstrings found in output)", "FAIL"))
except SyntaxError as e:
    checks_2.append((f"Adds docstrings (SyntaxError: {e})", "FAIL"))

# Test type hints
r2 = improve_code(starting_code, "type_hints")
try:
    compile(r2, "<string>", "exec")
    has_hints = "->" in r2
    if has_hints:
        checks_2.append(("Adds type hints (valid syntax + -> present)", "PASS"))
        score2 += 1
    else:
        checks_2.append(("Adds type hints (no -> found in output)", "FAIL"))
except SyntaxError as e:
    checks_2.append((f"Adds type hints (SyntaxError: {e})", "FAIL"))

# Test error handling
r3 = improve_code(starting_code, "error_handling")
try:
    compile(r3, "<string>", "exec")
    has_try = "try:" in r3 and "except" in r3
    if has_try:
        checks_2.append(("Adds error handling (valid syntax + try/except)", "PASS"))
        score2 += 1
    else:
        checks_2.append(("Adds error handling (no try/except found)", "FAIL"))
except SyntaxError as e:
    checks_2.append((f"Adds error handling (SyntaxError: {e})", "FAIL"))

for check, status in checks_2:
    print(f"    [{status}] {check}")

print(f"\n  Score: {score2}/3")

## TODO 3 Solution: Implement `iterative_refine`

In [ ]:
def iterative_refine(code, target_score=0.75, max_iterations=3):
    """Iteratively improve code until quality target is reached."""
    improvements = ["docstrings", "type_hints", "error_handling"]
    current_code = code
    history = []

    for i in range(min(max_iterations, len(improvements))):
        # Measure current quality
        before = code_quality_score(current_code)
        if before["overall"] >= target_score:
            break

        # Apply improvement
        improvement = improvements[i]
        current_code = improve_code(current_code, improvement)

        # Measure new quality
        after = code_quality_score(current_code)

        history.append({
            "round": i + 1,
            "improvement": improvement,
            "score_before": before["overall"],
            "score_after": after["overall"],
        })

    final = code_quality_score(current_code)

    return {
        "final_code": current_code,
        "iterations": len(history),
        "history": history,
        "final_score": final["overall"],
    }

In [ ]:
score3 = 0
checks_3 = []

r1 = iterative_refine(starting_code, target_score=0.75, max_iterations=3)

if isinstance(r1, dict) and "final_code" in r1 and "history" in r1:
    checks_3.append(("Returns result dict", "PASS"))
    score3 += 1
else:
    checks_3.append((f"Returns result dict (got keys: {list(r1.keys()) if isinstance(r1, dict) else '?'})", "FAIL"))

iters = r1.get("iterations", 0) if isinstance(r1, dict) else 0
if iters >= 2:
    checks_3.append((f"Ran multiple iterations ({iters})", "PASS"))
    score3 += 1
else:
    checks_3.append((f"Ran multiple iterations (got {iters})", "FAIL"))

history = r1.get("history", []) if isinstance(r1, dict) else []
if len(history) >= 1:
    first_before = history[0].get("score_before", 1)
    final_score = r1.get("final_score", 0) if isinstance(r1, dict) else 0
    if final_score > first_before:
        checks_3.append((f"Quality improved ({first_before:.2f} → {final_score:.2f})", "PASS"))
        score3 += 1
    else:
        checks_3.append((f"Quality improved (before={first_before}, after={final_score})", "FAIL"))
else:
    checks_3.append(("Quality improved (no history entries)", "FAIL"))

final_code = r1.get("final_code", "") if isinstance(r1, dict) else ""
try:
    compile(final_code, "<string>", "exec")
    checks_3.append(("Final code is valid Python", "PASS"))
    score3 += 1
except SyntaxError as e:
    checks_3.append((f"Final code is valid Python (SyntaxError: {e})", "FAIL"))

for check, status in checks_3:
    print(f"    [{status}] {check}")

# Print iteration history
print(f"\n  Refinement History:")
for entry in r1["history"]:
    print(f"    Round {entry['round']}: {entry['improvement']}"
          f"  score: {entry['score_before']:.2f} → {entry['score_after']:.2f}")

print(f"\n  Score: {score3}/4")

## Save Results

In [ ]:
# Save the refined code
with open(os.path.join(WORKDIR, "refined_module.py"), "w") as f:
    f.write(r1["final_code"])

# Save the refinement report
report = {
    "iterations": r1["iterations"],
    "final_score": r1["final_score"],
    "history": r1["history"],
}
with open(os.path.join(WORKDIR, "refinement-report.json"), "w") as f:
    json.dump(report, f, indent=2)

print(f"Refined code saved to {WORKDIR}/refined_module.py")
print(f"Report saved to {WORKDIR}/refinement-report.json")

## Lab 08 Summary

In [ ]:
total = score1 + score2 + score3
max_total = 4 + 3 + 4

print(f"  TODO 1: {score1}/4 quality scoring checks passed")
print(f"  TODO 2: {score2}/3 code improvement checks passed")
print(f"  TODO 3: {score3}/4 iterative refinement checks passed")
print(f"\n  Total: {total}/{max_total}")
print(f"\n  Files generated in {WORKDIR}/")

### Key Takeaways

1. **Measure first:** You can't improve what you don't measure — quality metrics drive the loop
2. **One fix per round:** Agents apply targeted improvements, not everything at once
3. **Stop conditions** prevent infinite loops — either hit the target or hit max iterations
4. **History tracking** lets you see what each round contributed